# Baseline C: TC-CLIP on UCF101 and HMDB51 (Kaggle)

Sixth Kaggle session; it can run at the same time as the HMDB51 notebook. TC-CLIP's text
embeddings depend on each video (video-conditional prompting), so it cannot run through this
repository's frozen-text adapter. It is scored by **TC-CLIP's official evaluation code**
(`main.py`, `eval=test`, zero-shot Kinetics-400 checkpoint), through `envs/tcclip_eval.py`. That
script runs the official code unchanged and only records each video's scores, because the
official code prints accuracies but saves no predictions. `ovar import-tcclip` then turns the
scores into a normal run: overall, familiar and unfamiliar accuracy with cluster-bootstrap
intervals.

1. **Gate:** TC-CLIP native on 2 videos per class. The saved scores must reproduce the accuracy
   the official code prints, so the per-video alignment is checked before the full runs.
2. **Native configuration** (`tcclip_native_16x2x1`): 16 frames × 2 randomly jittered views
   (seed 1024), learned prompts, as in TC-CLIP's own zero-shot test. Reported as Baseline C.
3. **Side analysis** (`tcclip_single_16x1x1`): the official code with multi-view inference off.
   This gives one deterministic view of 16 segment midpoints, the frame positions of the common
   protocol. TC-CLIP's own resize/crop and learned prompts remain, so it is not the common
   protocol.

Each dataset runs in its own invocation, so the random views of one dataset do not depend on the
other.

**Before running**

1. Settings → Accelerator: **GPU T4**. Settings → Internet: **On**.
2. **Add input:** datasets **lifailifai/ucf101** and **kodurujagruthaaditya/hmdb51**. Without the
   UCF101 dataset, section 3 downloads the official archive (6.9 GB). Without HMDB51, its stages
   are skipped.
3. Save Version → **Save & Run All (Commit)**. Download **`ovar_outputs_6.zip`** from the Output tab.

Rough time: setup 20 min, gate 5 min, native about 60 min, side analysis about 35 min. About 2 hours
in total.

In [ ]:
import os, subprocess, sys, json, hashlib, shutil
from pathlib import Path

REPO = "eylulpelinkilic/generalized-category-discovery"
REF = "main"            # replace with a commit SHA to pin the code version
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
PROJECT = CODE / "action_recognition"
RUN_NATIVE = True       # 2. TC-CLIP native (Baseline C)
RUN_SINGLE = True       # 3. side analysis: single deterministic view
ZIP = "ovar_outputs_6.zip"
STATUS = {}


def run(cmd, cwd=None, allow_fail=False, log=None):
    """Run a shell command, stream (and optionally save) its output, raise on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (PROJECT if PROJECT.exists() else WORK),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    sink = open(log, "a") if log else None
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-15:]
        if sink:
            sink.write(line)
    code = p.wait()
    if sink:
        sink.close()
    if code and not allow_fail:
        last = "\n".join(tail)
        raise RuntimeError(f"command failed with exit code {code}: {cmd}\n--- last output lines ---\n{last}")
    return code


def step(name, fn, needs=(), enabled=True):
    """Run one stage; record a failure and continue, so independent later stages still run."""
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if not enabled:
        STATUS[name] = "not requested"
        return
    if missing:
        STATUS[name] = f"skipped: needs {', '.join(missing)}"
        print(f"-- {name} skipped (needs {missing})")
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

## 1. Code: clone the repository

In [ ]:
if subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}", REF],
                  capture_output=True, text=True).returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Settings -> Internet on (phone-verified account), "
                       "then restart the session.")

if not CODE.exists():
    CODE.mkdir(parents=True)
    run("git init -q && git remote add origin https://github.com/" + REPO + ".git", cwd=CODE)
run(f"git fetch -q --depth 1 origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit:", COMMIT)
assert (PROJECT / "kaggle" / "ovar_tcclip_baselines.ipynb").exists(), \
    "the cloned code predates this notebook: push the latest commit to GitHub (or set REF)"

## 2. Environment (this repository)

In [ ]:
run("pip install -q open_clip_torch ftfy regex")
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")

import torch, yaml
print("python", sys.version.split()[0], "| torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU"

run("python -m ovar fetch-repos")
run("python -m unittest discover -s tests -t .")
TCC = PROJECT / "third_party" / "tc-clip"
TCC_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=TCC, capture_output=True, text=True).stdout.strip()
print("TC-CLIP commit:", TCC_COMMIT)

## 3. Data: UCF101 and HMDB51

UCF101 is found as in sessions 1–4 (attached copy, otherwise the official archive). HMDB51 is
found as in the HMDB51 notebook (attached copy only; the official link is dead). Validation then
checks every listed test file and decodes 100 of them.

In [ ]:
OFFICIAL_URL = "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar"
OFFICIAL_BYTES = 6_932_971_618       # Content-Length served by CRCV (Last-Modified 2012-11-21)
DOWNLOAD_DIR = Path("/tmp/ucf101")   # ephemeral disk, not the 20 GB /kaggle/working output


def find_ucf():
    found = [p.parent for depth in range(5)
             for p in Path("/kaggle/input").glob("*/" * depth + "UCF-101/ApplyEyeMakeup")]
    return sorted(found, key=lambda p: len(p.parts))


def download_ucf():
    root = DOWNLOAD_DIR / "UCF-101"
    rar = DOWNLOAD_DIR / "UCF101.rar"
    if (root / "ApplyEyeMakeup").is_dir() and not rar.exists():
        return root, json.loads((DOWNLOAD_DIR / "archive.json").read_text())
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DOWNLOAD_DIR).free
    if free < 2.3 * OFFICIAL_BYTES:
        raise RuntimeError(f"need about 16 GB free in {DOWNLOAD_DIR}, have {free / 1e9:.1f} GB")
    for attempt in range(3):         # curl resumes a partial file with -C -
        if rar.exists() and rar.stat().st_size == OFFICIAL_BYTES:
            break
        run(f"curl -sS -L --fail --retry 5 --retry-delay 10 -C - -o {rar} {OFFICIAL_URL}", allow_fail=True)
    size = rar.stat().st_size if rar.exists() else 0
    if size != OFFICIAL_BYTES:
        raise RuntimeError(f"download incomplete: {size} of {OFFICIAL_BYTES} bytes")
    archive = {"url": OFFICIAL_URL, "bytes": size, "sha256": sha256(rar)}
    run("apt-get -qq update", allow_fail=True)
    for tool, package, cmd in [("unrar", "unrar", f"unrar x -idq -o+ {rar} {DOWNLOAD_DIR}/"),
                               ("unar", "unar", f"unar -q -f -o {DOWNLOAD_DIR} {rar}"),
                               ("bsdtar", "libarchive-tools", f"bsdtar -xf {rar} -C {DOWNLOAD_DIR}")]:
        if shutil.which(tool) is None:
            run(f"apt-get -qq install -y {package}", allow_fail=True)
        if shutil.which(tool) and run(cmd, allow_fail=True) == 0 and (root / "ApplyEyeMakeup").is_dir():
            archive["extracted_with"] = tool
            break
    else:
        raise RuntimeError("could not extract UCF101.rar (tried unrar, unar, bsdtar)")
    (DOWNLOAD_DIR / "archive.json").write_text(json.dumps(archive) + "\n")
    rar.unlink()
    return root, archive


def find_hmdb():
    found = []
    for depth in range(5):
        for p in Path("/kaggle/input").glob("*/" * depth + "brush_hair"):
            if any(p.glob("*.avi")):
                found.append((p.parent, True))
            elif any(p.glob("brush_hair/*.avi")):
                found.append((p.parent, False))
    return sorted(found, key=lambda x: len(x[0].parts))


ROOTS, DATA_SOURCE, SINGLE_FOLDER = {}, {}, False
ucf = find_ucf()
if ucf:
    ROOTS["ucf101"] = ucf[0]
    DATA_SOURCE["ucf101"] = {"source": "kaggle input", "path": str(ucf[0])}
else:
    print("no attached UCF101 copy; downloading the official archive")
    root, archive = download_ucf()
    ROOTS["ucf101"] = root
    DATA_SOURCE["ucf101"] = {"source": "official CRCV archive", "path": str(root), **archive}
hmdb = find_hmdb()
if hmdb:
    ROOTS["hmdb51"], SINGLE_FOLDER = hmdb[0]
    DATA_SOURCE["hmdb51"] = {"source": "kaggle input", "path": str(hmdb[0][0]), "single_class_folder": hmdb[0][1]}
else:
    print("!! no HMDB51 copy attached (kodurujagruthaaditya/hmdb51): its stages are skipped")
DATASETS = list(ROOTS)
print(json.dumps(DATA_SOURCE, indent=2))
(PROJECT / "runs").mkdir(exist_ok=True)
(PROJECT / "runs" / "DATA_SOURCE.json").write_text(json.dumps(DATA_SOURCE, indent=2) + "\n")

TC_CKPT = Path("/tmp/checkpoints/tcclip/zero_shot_k400_tc_clip.pth")   # outside the outputs
CLIP_JIT = Path.home() / ".cache" / "clip" / "ViT-B-16.pt"   # where TC-CLIP's clip._download looks
(PROJECT / "configs" / "paths.yaml").write_text(
    "dataset_roots:\n" + "".join(f"  {k}: {v}\n" for k, v in ROOTS.items())
    + f"hmdb51_single_class_folder: {'true' if SINGLE_FOLDER else 'false'}\n"
    "checkpoints:\n"
    f"  tcclip_zero_shot: {TC_CKPT}\n"
    "cache_dir: cache\n"
    "runs_dir: runs\n")
print((PROJECT / "configs" / "paths.yaml").read_text())
for name in DATASETS:
    run(f"python -m ovar validate --datasets {name} --check-files --check-decode --decode-limit 100")

## 4. Weights

* TC-CLIP `zero_shot_k400_tc_clip.pth`, the **zero-shot** checkpoint trained on Kinetics-400 only
  (README "Zero-shot action recognition", Drive folder `1gAB3LtNzVerkVerifgJFRL0WciPvrAlR`). Not the
  few-shot or base-to-novel ones, which are trained on UCF101/HMDB51, and not the "(LLM)" variant.
  No checksum is published; the size is checked and the SHA-256 recorded.
* Original OpenAI `ViT-B-16.pt`, which TC-CLIP builds on. Its URL contains its SHA-256, which is
  verified.

In [ ]:
TC_URL = ("https://drive.usercontent.google.com/download?id=1H8kHJGKIBuhHWlJLX5klaXtmb0Mw1tLp"
          "&export=download&confirm=t")
TC_BYTES = 531_363_279
CLIP_JIT_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_JIT_URL = f"https://openaipublic.azureedge.net/clip/models/{CLIP_JIT_SHA}/ViT-B-16.pt"
CLIP_JIT_BYTES = 350_837_078
DOWNLOADS = {}


def fetch(url, dest, nbytes):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not (dest.exists() and dest.stat().st_size == nbytes):
        run(f"curl -sS -L --fail --retry 5 -C - -o '{dest}' '{url}'")
    size = dest.stat().st_size
    if size != nbytes:
        raise RuntimeError(f"{dest.name}: got {size} bytes, expected {nbytes}. If Google Drive refuses "
                           "(download quota), download the file in a browser, upload it as a private "
                           f"Kaggle dataset, and copy it to {dest}.")
    return {"url": url, "bytes": size, "sha256": sha256(dest)}


def weights():
    DOWNLOADS["tcclip_zero_shot"] = fetch(TC_URL, TC_CKPT, TC_BYTES)
    DOWNLOADS["clip_vitb16_openai_pt"] = fetch(CLIP_JIT_URL, CLIP_JIT, CLIP_JIT_BYTES)
    print(json.dumps(DOWNLOADS, indent=2))
    (PROJECT / "runs" / "DOWNLOADS.json").write_text(json.dumps(DOWNLOADS, indent=2) + "\n")
    if DOWNLOADS["clip_vitb16_openai_pt"]["sha256"] != CLIP_JIT_SHA:
        raise RuntimeError("ViT-B-16.pt SHA-256 does not match its official URL")


step("weights", weights)

## 5. Environment for the official TC-CLIP code

TC-CLIP's INSTALL.md: Python 3.8, PyTorch 1.13.0 / torchvision 0.14.0, `requirements.txt`, and
NVIDIA Apex with its `cached_cast` replaced by `scripts/apex_custom_cached_cast.py`. This builds it
in a separate virtual environment (`uv` provides Python 3.8). Deviations, none of which touches
evaluation:

* **PyTorch from the official pip wheels** (`+cu117`) instead of conda. TC-CLIP decodes with
  decord, not torchvision's video reader.
* **Apex at a pinned commit** (`810ffae`, the one INSTALL.md links) and **Python-only**.
  `apex.amp` (O1 mixed precision, used at test time) is pure Python. The compiled extensions are
  fused training kernels, and they would need a CUDA 11.7 compiler that Kaggle does not have.
* **`pathlib==1.0.1` is skipped:** it is an obsolete Python 2 backport, and pathlib is in the
  standard library.
* **`mmcv-full==1.7.0`** comes from OpenMMLab's prebuilt wheel index for torch 1.13 / cu117, and
  **numpy is pinned to 1.23.5**. requirements.txt leaves numpy unpinned.

In [ ]:
TC_ENV = Path("/tmp/tcclip-env")
TC_PY = TC_ENV / "bin" / "python"
APEX = Path("/tmp/apex")
APEX_COMMIT = "810ffae374a2b9cb4b5c5e28eaeca7d7998fca0c"
TC_REQUIREMENTS = ["mmcv-full==1.7.0", "decord==0.6.0", "ftfy==6.1.1", "einops==0.6.0", "termcolor==2.1.1",
                   "timm==0.4.12", "regex==2022.10.31", "yacs==0.1.8", "pandas==1.3.5", "hydra-core==1.3.2",
                   "wandb==0.16.2", "scikit-image==0.21.0"]
TC_CONSTRAINTS = Path("/tmp/tcclip-constraints.txt")


def tcclip_env():
    TC_CONSTRAINTS.write_text("torch==1.13.0+cu117\ntorchvision==0.14.0+cu117\nnumpy==1.23.5\n")
    run("pip install -q uv")
    run("uv python install 3.8")
    if not TC_PY.exists():
        run(f"uv venv --python 3.8 --seed {TC_ENV}")
    pip = f"{TC_PY} -m pip install -q -c {TC_CONSTRAINTS}"
    run(f"{pip} --extra-index-url https://download.pytorch.org/whl/cu117 torch torchvision numpy")
    reqs = " ".join(f"'{r}'" for r in TC_REQUIREMENTS)
    run(f"{pip} -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13/index.html {reqs}")

    if not APEX.exists():
        run(f"git init -q {APEX} && git -C {APEX} remote add origin https://github.com/NVIDIA/apex.git")
        run(f"git -C {APEX} fetch -q --depth 1 origin {APEX_COMMIT} && git -C {APEX} checkout -q FETCH_HEAD")
    utils = APEX / "apex" / "amp" / "utils.py"
    source = utils.read_text()
    custom = (TCC / "scripts" / "apex_custom_cached_cast.py").read_text()
    replacement = custom[custom.index("def cached_cast("):].rstrip() + "\n\n"
    if replacement not in source:            # INSTALL.md step 2
        start = source.index("def cached_cast(cast_fn, x, cache):")
        end = source.index("def verbosify(", start)
        utils.write_text(source[:start] + replacement + source[end:])
    run(f"{TC_PY} -m pip install -q --no-build-isolation {APEX}")

    run(f"{TC_PY} -c \"import torch, torchvision, numpy, mmcv, decord, hydra, timm; from apex import amp; "
        f"print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'numpy', numpy.__version__, "
        f"'mmcv', mmcv.__version__, 'decord', decord.__version__, 'timm', timm.__version__); "
        f"print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); torch.zeros(1).cuda()\"")
    run(f"{TC_PY} -c \"import main, engine; print('official TC-CLIP code imports')\"", cwd=TCC)
    run(f"{TC_PY} -m pip freeze > {PROJECT / 'runs' / 'tcclip_env_freeze.txt'}")


step("tcclip_env", tcclip_env, needs=["weights"])

## 6. Annotation files and data configs

`export-native --backbone tcclip` writes each test list (`<path> <label>`) and label file
(`id,name`) with our class texts in XOV's label order. A Hydra data config per dataset is added
inside the TC-CLIP clone. Its `train`/`val` entries are the unused Kinetics placeholders that the
official configs carry, and `test` holds exactly one dataset. The dataset name keeps `ucf101` /
`hmdb51`, so the official loader settings for these datasets apply (main-process loading).

In [ ]:
TC_OUT = PROJECT / "runs" / "tcclip_official"
EXPORT = TC_OUT / "export"
EXPORT_GATE = TC_OUT / "export_gate"
PROTOCOLS = {p: yaml.safe_load((PROJECT / "configs" / "protocols" / f"{p}.yaml").read_text())
             for p in ("tcclip_native_16x2x1", "tcclip_single_16x1x1")}
N_CLASSES = {"ucf101": 101, "hmdb51": 51}
GATE_DATASET = DATASETS[0]


def data_config(config_name, name, export):
    text = (
        "#@package _global_\n"
        "data:\n"
        "  train: {dataset_name: k400_train, root: ${k400.root}, num_classes: 400, "
        "label_file: ${k400.label_files.default}, ann_file: ${k400.ann_files.train}}\n"
        "  val: {dataset_name: k400_val, root: ${k400.root}, num_classes: 400, "
        "label_file: ${k400.label_files.default}, ann_file: ${k400.ann_files.val}}\n"
        "  test:\n"
        f"    - name: {name}\n"
        "      protocol: top1\n"
        "      dataset_list:\n"
        f"        - {{dataset_name: {name}, root: {ROOTS[name]}, num_classes: {N_CLASSES[name]}, "
        f"label_file: {export / name / f'{name}_labels.csv'}, ann_file: {export / name / 'test.txt'}}}\n")
    (TCC / "configs" / "data" / f"{config_name}.yaml").write_text(text)


def exports():
    run(f"python -m ovar export-native --backbone tcclip --datasets {' '.join(DATASETS)} --out {EXPORT}")
    run(f"python -m ovar export-native --backbone tcclip --datasets {GATE_DATASET} --per-class 2 "
        f"--out {EXPORT_GATE}")
    for name in DATASETS:
        data_config(f"xov_{name}", name, EXPORT)
        run(f"head -2 {EXPORT / name / 'test.txt'}")
    data_config(f"xov_{GATE_DATASET}_gate", GATE_DATASET, EXPORT_GATE)


step("exports", exports, needs=["tcclip_env"])

## 7. Official evaluation, one dataset per invocation

`envs/tcclip_eval.py` runs `torchrun main.py -cn zero_shot data=<config> trainer=tc_clip eval=test
resume=<checkpoint> gather_filename=true`, plus the protocol's `official` overrides, with
`main.main()` and `engine.validate` unchanged.

In [ ]:
PROVENANCE = PROJECT / "runs" / "tcclip_provenance.json"


def tcclip_eval(protocol, data_config_name, out):
    official = PROTOCOLS[protocol]["official"]
    overrides = " ".join(f"{k}={str(v).lower() if isinstance(v, bool) else v}" for k, v in official.items())
    out.mkdir(parents=True, exist_ok=True)
    run(f"{TC_PY} -m torch.distributed.run --nproc_per_node=1 --master_port 29517 "
        f"{PROJECT / 'envs' / 'tcclip_eval.py'} --repo {TCC} -- -cn zero_shot data={data_config_name} "
        f"trainer=tc_clip eval=test resume={TC_CKPT} output={out} gather_filename=true {overrides}",
        log=out / "console.log")


def import_scores(protocol, name, out, variant, per_class=None):
    scores = out / "scores"
    if not (scores / f"{name}.npz").exists():
        raise RuntimeError(f"no scores saved in {scores}")
    if not PROVENANCE.exists():
        torch_version = subprocess.run([str(TC_PY), "-c", "import torch; print(torch.__version__)"],
                                       capture_output=True, text=True).stdout.strip()
        PROVENANCE.write_text(json.dumps({
            "code": f"github.com/naver-ai/tc-clip @ {TCC_COMMIT}", "licence": "CC BY-NC 4.0",
            "checkpoint": str(TC_CKPT), "checkpoint_sha256": DOWNLOADS["tcclip_zero_shot"]["sha256"],
            "checkpoint_bytes": DOWNLOADS["tcclip_zero_shot"]["bytes"],
            "clip_sha256": DOWNLOADS["clip_vitb16_openai_pt"]["sha256"],
            "training_data": "Kinetics-400 only (zero-shot checkpoint)",
            "environment": {"device": "cuda", "torch": torch_version, "python": "3.8",
                            "gpu": torch.cuda.get_device_name(0)},
            "scored_by": "official main.py/engine.validate via envs/tcclip_eval.py"}, indent=2) + "\n")
    subset = f"--per-class {per_class} --n-boot 200" if per_class else ""
    run(f"python -m ovar import-tcclip tcclip_vitb16__{protocol} --set 'datasets=[{name}]' "
        f"--scores-dir {scores} --provenance {PROVENANCE} --variant {variant} {subset}")
    record = json.loads((PROJECT / "runs" / f"tcclip_vitb16__{protocol}__{variant}" / "run.json").read_text())
    meta = record["official_scores"][name]
    print(f"{name}: official Acc@1 {meta['official_acc1']:.3f} | recomputed from saved scores "
          f"{meta['recomputed_acc1']:.3f}")
    return meta


def gate():
    out = TC_OUT / "gate" / GATE_DATASET
    tcclip_eval("tcclip_native_16x2x1", f"xov_{GATE_DATASET}_gate", out)
    meta = import_scores("tcclip_native_16x2x1", GATE_DATASET, out, f"{GATE_DATASET}_subset2", per_class=2)
    n = 2 * N_CLASSES[GATE_DATASET]
    if abs(meta["official_acc1"] - meta["recomputed_acc1"]) > 100 / n + 1e-6:
        raise RuntimeError("saved scores do not reproduce the official accuracy: alignment problem")


step("gate", gate, needs=["exports"])
for protocol, enabled in (("tcclip_native_16x2x1", RUN_NATIVE), ("tcclip_single_16x1x1", RUN_SINGLE)):
    for name in DATASETS:
        out = TC_OUT / protocol / name
        step(f"{protocol}__{name}",
             lambda protocol=protocol, name=name, out=out: (tcclip_eval(protocol, f"xov_{name}", out),
                                                            import_scores(protocol, name, out, name)),
             needs=["gate"], enabled=enabled)

## 8. Reports and outputs

In [ ]:
step("report", lambda: run("python -m ovar report"))
if STATUS.get("report") == "ok":
    from IPython.display import Markdown, display
    display(Markdown((PROJECT / "reports" / "results.md").read_text()))

(PROJECT / "runs" / "CODE_COMMIT.txt").write_text(COMMIT + "\n")
(PROJECT / "runs" / "NOTEBOOK_STATUS.json").write_text(json.dumps(STATUS, indent=2) + "\n")
run(f"rm -f {WORK}/{ZIP} && cd {PROJECT} && zip -qr {WORK}/{ZIP} runs reports configs/paths.yaml")
run(f"ls -lh {WORK}/{ZIP}")

In [ ]:
# Does not raise: an error here could cost the saved outputs of a committed run.
failed = {k: v for k, v in STATUS.items() if v not in ("ok", "not requested")}
print("=" * 70)
print("STAGES NOT COMPLETED - send the zip or this output:" if failed else "all requested stages completed")
for name, state in STATUS.items():
    print(f"{name:36s} {state}")
print("=" * 70)